# Проверка заполненности тарифного плана

Тетрадка считает покрытие тарифного плана у клиентов за выбранный месяц
и выводит список договоров без найденного тарифа.

In [ ]:
# Параметр периода
month_start = "2026-02-01"  # формат YYYY-MM-01

In [ ]:
# 1) Сводка покрытия тарифного плана
with imp:
    tariff_cov = imp.fetch(f"""
        with params as (
            select cast('{month_start}' as date) as month_start
        ),
        feb_clients as (
            select distinct
                c.c_inn as inn,
                a.abs_agr_id as agr_id,
                a.c_agr_number
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            join ods_alpha.scd1_trx_acq ta
              on ta.n_agr = a.n_agr
            join ods_alpha.scd1_trx t
              on t.n_trx = ta.n_trx
            join params p on 1=1
            where a.acq_class = 'SA'
              and a.abs_agr_id is not null
              and t.c_trx_class = 'SA'
              and t.c_trx_type = 'S01'
              and t.c_nter is not null
              and t.ods_deleted_flg <> '1'
              and t.cf_trx_stat <> 'R'
              and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = p.month_start
        ),
        tariff_map as (
            select
                c.c_inn as inn,
                m.id as agr_id,
                tp.id as tariff_plan_id,
                tp.c_name as tariff_plan_name
            from ods.scd1_z_client c
            left join ods.scd1_z_r2_ip_merchants m
              on m.c_cl_org = c.id
            left join ods.scd1_z_r2_tariff_plan tp
              on tp.id = m.c_r2_tariff_plan
        )
        select
            count(*) as feb_agr_cnt,
            sum(case when tm.tariff_plan_id is not null then 1 else 0 end) as tariff_filled_cnt,
            sum(case when tm.tariff_plan_id is null then 1 else 0 end) as tariff_null_cnt,
            round(
              100.0 * sum(case when tm.tariff_plan_id is not null then 1 else 0 end) / count(*),
              2
            ) as tariff_fill_rate_pct
        from feb_clients fc
        left join tariff_map tm
          on tm.agr_id = fc.agr_id
    """)
tariff_cov

In [ ]:
# 2) Список договоров без найденного тарифного плана
with imp:
    tariff_nulls = imp.fetch(f"""
        with params as (
            select cast('{month_start}' as date) as month_start
        ),
        feb_clients as (
            select distinct
                c.c_inn as inn,
                a.abs_agr_id as agr_id,
                a.c_agr_number
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            join ods_alpha.scd1_trx_acq ta
              on ta.n_agr = a.n_agr
            join ods_alpha.scd1_trx t
              on t.n_trx = ta.n_trx
            join params p on 1=1
            where a.acq_class = 'SA'
              and a.abs_agr_id is not null
              and t.c_trx_class = 'SA'
              and t.c_trx_type = 'S01'
              and t.c_nter is not null
              and t.ods_deleted_flg <> '1'
              and t.cf_trx_stat <> 'R'
              and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = p.month_start
        ),
        tariff_map as (
            select
                m.id as agr_id,
                tp.id as tariff_plan_id,
                tp.c_name as tariff_plan_name
            from ods.scd1_z_r2_ip_merchants m
            left join ods.scd1_z_r2_tariff_plan tp
              on tp.id = m.c_r2_tariff_plan
        )
        select
            fc.inn,
            fc.c_agr_number,
            fc.agr_id
        from feb_clients fc
        left join tariff_map tm
          on tm.agr_id = fc.agr_id
        where tm.tariff_plan_id is null
        order by fc.inn, fc.c_agr_number
        limit 500
    """)
tariff_nulls